In [12]:
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install pandas numpy tables matplotlib --quiet

In [13]:
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, BatchNorm, GatedGraphConv
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import os
import zipfile

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [14]:
zip_path = 'METR-LA.zip'
data_dir = 'METR-LA'
if not os.path.exists(data_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
print(f"Data directory {data_dir} ready")

Data directory METR-LA ready


In [16]:
def load_metr_la_data(data_dir='METR-LA'):
    adj_mat = np.load(os.path.join(data_dir, 'adj_mat.npy'))  # Shape: [207, 207]
    node_values = np.load(os.path.join(data_dir, 'node_values.npy'))  # Shape: [34272, 207, 2]
    print(f"Original node_values shape: {node_values.shape}")
    if node_values.ndim == 3:
        node_values = node_values[:, :, 0]  # Shape: [34272, 207]
        node_values = node_values.T  # Shape: [207, 34272]
    elif node_values.ndim != 2:
        raise ValueError(f"Expected 2D or 3D node_values, got shape {node_values.shape}")
    print(f"Processed node_values shape: {node_values.shape}")
    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T  # Shape: [207, 34272]
    edge_index = np.where(adj_mat > 0)
    edge_index = torch.tensor(np.array(edge_index), dtype=torch.long)  # Keep on CPU
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)  # Keep on CPU
    # Move to device after all NumPy operations
    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)
    print(f"edge_index device: {edge_index.device}, shape: {edge_index.shape}")
    print(f"edge_weight device: {edge_weight.device}, shape: {edge_weight.shape}")
    return node_values, edge_index, edge_weight, scaler

# Run the function
node_values, edge_index, edge_weight, scaler = load_metr_la_data()

Original node_values shape: (34272, 207, 2)
Processed node_values shape: (207, 34272)
edge_index device: cuda:0, shape: torch.Size([2, 1722])
edge_weight device: cuda:0, shape: torch.Size([1722])


In [17]:
def create_data_samples(node_values, in_timestep=12, out_timestep=12):
    features, labels = [], []
    for i in range(node_values.shape[1] - in_timestep - out_timestep + 1):
        features.append(node_values[:, i:i + in_timestep])
        labels.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])
    features = torch.tensor(features, dtype=torch.float).permute(0, 2, 1)
    labels = torch.tensor(labels, dtype=torch.float).permute(0, 2, 1)
    return features, labels

# Run the function
features, labels = create_data_samples(node_values)

In [18]:
transform = AddRandomWalkPE(walk_length=20, attr_name='pe')
data_list = []
for i in range(features.shape[0]):
    x = features[i].T  # Shape: [207, 12]
    y = labels[i].T    # Shape: [207, 12]
    data = Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_weight)
    data = transform(data)
    data_list.append(data)

In [19]:
train_data = data_list[:int(0.7 * len(data_list))]
val_data = data_list[int(0.7 * len(data_list)):int(0.9 * len(data_list))]
test_data = data_list[int(0.9 * len(data_list)):]

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [20]:
class GPSLayer(nn.Module):
    def __init__(self, hidden_channels, heads=4):
        super(GPSLayer, self).__init__()
        self.gated_gcn = GatedGraphConv(hidden_channels, num_layers=1)
        self.transformer = TransformerConv(hidden_channels, hidden_channels, heads=heads, concat=False, dropout=0.1)
        self.norm = BatchNorm(hidden_channels)

    def forward(self, x, edge_index, edge_weight, batch):
        x_gcn = self.gated_gcn(x, edge_index, edge_weight=edge_weight)
        x_trans = self.transformer(x_gcn, edge_index)
        x = self.norm(x_gcn + x_trans)
        x = torch.relu(x)
        return x

In [21]:
class GPSGraphTransformer(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, pe_dim=20, heads=4):
        super().__init__()
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim)
        self.pe_lin = nn.Linear(pe_dim, pe_dim)
        self.layers = nn.ModuleList([
            GPSLayer(hidden_channels, heads) for _ in range(num_layers)
        ])
        self.norm = BatchNorm(hidden_channels)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, out_channels)
        )

    def forward(self, x, edge_index, edge_weight, pe, batch):
        x = self.node_emb(x)
        pe = self.pe_lin(pe)
        x = torch.cat([x, pe], dim=-1)
        for layer in self.layers:
            x = layer(x, edge_index, edge_weight, batch)
        x = self.norm(x)
        x = self.mlp(x)
        batch_size = batch.max().item() + 1
        num_nodes = x.size(0) // batch_size
        return x.view(batch_size, num_nodes, -1)

In [26]:
def compute_metrics(pred, target, scaler):
    # Shapes: pred, target = (batch_size, num_nodes, out_timestep), e.g., (32, 207, 12)
    batch_size, num_nodes, out_timestep = pred.shape

    # Reshape to (batch_size * out_timestep, num_nodes) for scaling
    pred_np = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)  # Shape: (batch_size * out_timestep, 207)
    target_np = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)  # Shape: (batch_size * out_timestep, 207)

    # Inverse transform
    pred_unscaled = scaler.inverse_transform(pred_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)

    # Convert back to tensors
    pred_unscaled = torch.tensor(pred_unscaled, device=pred.device)
    target_unscaled = torch.tensor(target_unscaled, device=target.device)

    # Compute metrics
    mae = torch.mean(torch.abs(pred_unscaled - target_unscaled)).item()
    rmse = torch.sqrt(torch.mean((pred_unscaled - target_unscaled) ** 2)).item()
    mask = target_unscaled > 1.0
    mape = torch.where(mask, torch.abs((pred_unscaled - target_unscaled) / (target_unscaled + 1e-2)), 0.0)
    mape = torch.mean(mape[mask]) * 100 if mask.sum() > 0 else 0.0
    return mae, rmse, mape

In [23]:
def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=5):
    os.makedirs(output_dir, exist_ok=True)
    pred = pred[0].cpu().numpy().T
    target = target[0].cpu().numpy().T
    pred = scaler.inverse_transform(pred).T
    target = scaler.inverse_transform(target).T
    plt.figure(figsize=(12, 10))
    for i in range(min(num_sensors, pred.shape[0])):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred[i], label='Predicted', marker='o')
        plt.plot(target[i], label='Actual', marker='x')
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'prediction_plot.png'))
    plt.close()

In [24]:
model = GPSGraphTransformer(
    in_channels=12, hidden_channels=64, out_channels=12, num_layers=4, pe_dim=20, heads=4
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
criterion = nn.MSELoss()

for epoch in range(20):
    model.train()
    total_train_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
        batch_size = data.batch.max().item() + 1
        num_nodes = out.shape[1]
        target = data.y.view(batch_size, num_nodes, -1)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            loss = criterion(out, target)
            total_val_loss += loss.item()
    val_loss = total_val_loss / len(val_loader)
    print(f"Epoch {epoch + 1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

Epoch 1, Train Loss: 0.0396, Val Loss: 0.0313
Epoch 2, Train Loss: 0.0233, Val Loss: 0.0310
Epoch 3, Train Loss: 0.0227, Val Loss: 0.0296
Epoch 4, Train Loss: 0.0225, Val Loss: 0.0288
Epoch 5, Train Loss: 0.0223, Val Loss: 0.0297
Epoch 6, Train Loss: 0.0220, Val Loss: 0.0292
Epoch 7, Train Loss: 0.0215, Val Loss: 0.0277
Epoch 8, Train Loss: 0.0216, Val Loss: 0.0277
Epoch 9, Train Loss: 0.0211, Val Loss: 0.0272
Epoch 10, Train Loss: 0.0209, Val Loss: 0.0278
Epoch 11, Train Loss: 0.0206, Val Loss: 0.0276
Epoch 12, Train Loss: 0.0205, Val Loss: 0.0273
Epoch 13, Train Loss: 0.0203, Val Loss: 0.0277
Epoch 14, Train Loss: 0.0202, Val Loss: 0.0269
Epoch 15, Train Loss: 0.0200, Val Loss: 0.0270
Epoch 16, Train Loss: 0.0199, Val Loss: 0.0272
Epoch 17, Train Loss: 0.0198, Val Loss: 0.0267
Epoch 18, Train Loss: 0.0196, Val Loss: 0.0269
Epoch 19, Train Loss: 0.0197, Val Loss: 0.0299
Epoch 20, Train Loss: 0.0195, Val Loss: 0.0269


In [27]:
model.eval()
total_mae, total_rmse, total_mape = 0, 0, 0
num_batches = 0
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
        batch_size = data.batch.max().item() + 1
        num_nodes = out.shape[1]
        target = data.y.view(batch_size, num_nodes, -1)
        mae, rmse, mape = compute_metrics(out, target, scaler)
        total_mae += mae
        total_rmse += rmse
        total_mape += mape
        num_batches += 1
        if num_batches == 1:
            visualize_predictions(out, target, scaler)

print(f"Test MAE: {total_mae / num_batches:.4f} mph")
print(f"Test RMSE: {total_rmse / num_batches:.4f} mph")
print(f"Test MAPE: {total_mape / num_batches:.4f}%")

Test MAE: 6.0468 mph
Test RMSE: 9.7566 mph
Test MAPE: 11.3013%
